<a href="https://colab.research.google.com/github/23151651/ISYS2001-23151651/blob/main/A2Explaination2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#This is the section that connects my establised basic code to Gemini AI
#I heavily had help from Chat GPT because this was the 2nd hardest bit of the asssessment.
#

In [ ]:
#Gemini API import from colab's secrets
import requests
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GEMINI_MODEL = "gemini-flash-latest"
#This is how the AI is like, personalized and told what to do.
FINANCE_PERSONA = """
you are a friendly, supportive and sassy Personal Finance Assistant.
Your purpose is to help users learn about their income, expenses, budgets, saving habits and spending habits.
When financial data is provided, use the actual figures provided. Do not make up transactions, financial figures or user circumstances.
Explain the calculations clearly to the user and give realistic solutions.
If the user asks something unrelated, sass them to be on topic and tell the user you are a finance assistant.
If a question is unclear, ask the user to clarify. Also don't say your a liscensed financial advisor.
"""
#This part is for asking gemini a financial question and giving an answer in it's programmed persona.
#Due to how I put in the persona it'll give it to you in a sassy way.
#I asked chat gpt: "How do I code gemini to recieve and answer questions, here's what i have so far: [definitions]"
#From there I read the difference ifs and defining I had to do to make it work.

def ask_gemini(question, financial_summary=None):
  """Send a finance question to Gemini and return its answer."""
  if not isinstance(question, str) or not question.strip():
    return "Enter a question."
  if not GEMINI_API_KEY:
    return "Error: GEMINI Key not found, maybe you dropped it :P"
  if financial_summary is not None:
    prompt = f"""
User's financial data:
{financial_summary}
User's question:
{question}
"""
#This is the part where it tells the user when there has been an error.
#I asked chat gpt: "what erros do i need to define if there is one in the code"
#It told me the 4 errors that I've put down.

  else:
    prompt = question
  url = ("https://generativelanguage.googleapis.com/v1beta/"f"models/{GEMINI_MODEL}:generateContent")
  headers = {"x-goog-api-key": GEMINI_API_KEY, "Content-Type": "application/json"}
  payload = {"system_instruction":{"parts":[{"text": FINANCE_PERSONA}]}, "contents": [{"parts": [{"text": prompt}]}]}
  try:
    response = requests.post(url, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()
    answer = data["candidates"][0]["content"]["parts"][0]["text"]
    return answer
  except requests.exceptions.Timeout:
    return "The request timed out. Please try again."
  except requests.exceptions.HTTPError as error:
    return (f"Gemini API error: {error}. Check the API key, availability and/or the quota. ")
  except (KeyError, IndexError, TypeError):
    return "Gemini returned an unexpected response. Please try again."
  except requests.exceptions.RequestException:
    return "Unable to connect to Gemini. Please check your connection."


In [ ]:
#This is a crude AI chat box to ask financial advise from Gemini AI.
#This is just to make sure it works
answer = ask_gemini("Explain 3 ways someone can manage expenses.")
print(answer)

In [ ]:
#This is the part where Gemini can analyze the given data from the start
#Instead of the plain hard coded analysis from my original and plain financial tracker, Gemini can do it for you
#And you can ask questions about the budget.
#I had AI tell me I needed to add an example_df for Gemini to know what it was analyzing.

example_df = pd.DataFrame([
    {"Type": "Income", "Amount": 3000.0, "Category": "Salary"},
    {"Type": "Expense", "Amount": 800.0, "Category": "Rent"},
    {"Type": "Expense", "Amount": 250.0, "Category": "Groceries"},
    {"Type": "Expense", "Amount": 100.0, "Category": "Transport"}])
summary = calculate_budget_breakdown(example_df, 1500)
answer = ask_gemini("Am I staying within my monthly budget? What's my position.", financial_summary=summary)
print(answer)